# 📡 From Fourier to Wavelets: A Practical Guide for Time-Frequency Analysis
### *Prepared for the Haemograph Capstone Project — Wavelet-Based Rheological Analysis*

---

**What this notebook covers:**
1. Why we need frequency analysis at all
2. Fourier Transform — the classic approach and its limitations
3. Short-Time Fourier Transform (STFT) — the band-aid fix
4. Wavelet Transform — the upgrade
5. Types of wavelets and when to use each
6. How all of this connects to your capstone (gel gelation, blood coagulation)

**Estimated reading + running time:** ~30 minutes  
**Prerequisites:** Basic Python, basic understanding of sine waves

---

In [1]:
# ============================================================
# SETUP: Install and import everything we need
# Run this cell first!
# ============================================================

# If you don't have PyWavelets installed, uncomment the line below:
# !pip install PyWavelets

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.fft import fft, fftfreq
import pywt  # PyWavelets — the main wavelet library
import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("✅ All libraries loaded successfully!")
print(f"PyWavelets version: {pywt.__version__}")

✅ All libraries loaded successfully!
PyWavelets version: 1.8.0


---
## 🧩 Part 1: Why Do We Need Frequency Analysis?

Imagine you're measuring a gel that's solidifying. Your rheometer records stress and strain values over time. The raw signal looks like a wiggly line — but hidden inside that wiggly line is information about:

- **How stiff** the gel is becoming (low frequency behaviour)
- **How elastic vs viscous** it is (phase angle)
- **When transitions happen** (structural changes mid-experiment)

To extract this, we need to **decompose the signal into its frequency components**.  
Think of it like a prism splitting white light into colours — each colour is a frequency.

Let's start by building an intuition with simple signals.

In [ ]:
# ============================================================
# BUILDING INTUITION: What is a signal made of?
# 
# Any real-world signal can be thought of as a sum of
# simple sine waves at different frequencies.
# Let's build one from scratch.
# ============================================================

# Time axis: 2 seconds, sampled at 1000 Hz (1000 points per second)
fs = 1000          # sampling frequency in Hz
t = np.linspace(0, 2, 2 * fs, endpoint=False)

# Three pure sine wave components
freq1, freq2, freq3 = 5, 20, 50   # Hz
amp1,  amp2,  amp3  = 1.0, 0.5, 0.3

comp1 = amp1 * np.sin(2 * np.pi * freq1 * t)   # slow oscillation (5 Hz)
comp2 = amp2 * np.sin(2 * np.pi * freq2 * t)   # medium (20 Hz)
comp3 = amp3 * np.sin(2 * np.pi * freq3 * t)   # fast (50 Hz)

# The combined signal — this is what a sensor would record
combined = comp1 + comp2 + comp3

# Plot the components and the combined signal
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('A signal is a sum of sine waves at different frequencies', fontsize=14, fontweight='bold')

axes[0].plot(t[:500], comp1[:500], color='steelblue');  axes[0].set_ylabel('5 Hz'); axes[0].set_ylim(-1.5, 1.5)
axes[1].plot(t[:500], comp2[:500], color='darkorange'); axes[1].set_ylabel('20 Hz'); axes[1].set_ylim(-1.5, 1.5)
axes[2].plot(t[:500], comp3[:500], color='green');      axes[2].set_ylabel('50 Hz'); axes[2].set_ylim(-1.5, 1.5)
axes[3].plot(t[:500], combined[:500], color='purple');  axes[3].set_ylabel('Combined'); axes[3].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

print("👆 Notice: the combined signal (purple) looks complex — but it's just 3 sine waves added together.")
print("   The challenge of signal analysis is to 'unmix' this back into its components.")

---
## 📐 Part 2: The Fourier Transform (FT)

### The Big Idea

The Fourier Transform answers the question:  
> *"What frequencies are present in this signal, and how strong is each one?"*

**How it works mathematically:**  
It multiplies your signal by sine waves of every possible frequency and integrates (sums) the result.  
If your signal contains a 5 Hz component, the multiplication with a 5 Hz sine wave gives a big result.  
If it doesn't contain 50 Hz, that multiplication gives nearly zero.

$$X(f) = \int_{-\infty}^{\infty} x(t) \cdot e^{-j2\pi ft} \, dt$$

Don't worry about the formula — the intuition is: **correlation with sine waves at each frequency**.

### The Catch — and it's a BIG one

The Fourier Transform assumes the signal is **stationary** — i.e., the frequencies present don't change over time.  
It gives you a **global** picture: "this signal contains 5 Hz" — but NOT "5 Hz appeared at t=1.2 seconds".

For gelation experiments, this is a serious problem. The gel *evolves* — its frequency content changes!

In [ ]:
# ============================================================
# FOURIER TRANSFORM IN ACTION
#
# Let's apply FFT (Fast Fourier Transform — the digital version
# of FT) to our combined signal and recover the 3 frequencies.
# ============================================================

# Compute the FFT
N = len(combined)
yf = fft(combined)                  # complex frequency coefficients
xf = fftfreq(N, 1 / fs)             # corresponding frequency values

# We only care about positive frequencies (the spectrum is mirrored)
half = N // 2
magnitude = (2 / N) * np.abs(yf[:half])   # amplitude spectrum

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Time domain
axes[0].plot(t[:500], combined[:500], color='purple', linewidth=0.8)
axes[0].set_title('Time Domain Signal (first 0.5 seconds)')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')

# Frequency domain — the FFT output
axes[1].plot(xf[:half], magnitude, color='crimson', linewidth=1.5)
axes[1].set_xlim(0, 80)
axes[1].set_title('Frequency Domain (FFT Output)')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Amplitude')

# Annotate the three peaks
for f, a, label in [(5, amp1, '5 Hz\n(amp=1.0)'), (20, amp2, '20 Hz\n(amp=0.5)'), (50, amp3, '50 Hz\n(amp=0.3)')]:
    axes[1].annotate(label, xy=(f, a), xytext=(f+3, a+0.05),
                     arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

plt.tight_layout()
plt.show()

print("✅ FFT correctly identified all three frequencies (5, 20, 50 Hz) and their amplitudes.")
print("   But notice: the FFT gives NO information about WHEN these frequencies occur.")

In [ ]:
# ============================================================
# THE CRITICAL PROBLEM WITH FFT: Time-Blindness
#
# Let's create TWO very different signals that produce
# IDENTICAL FFT outputs. This shows why FFT alone fails
# for evolving systems like gels.
# ============================================================

t2 = np.linspace(0, 4, 4 * fs, endpoint=False)

# Signal A: 5 Hz and 20 Hz present throughout (stationary)
signal_A = (np.sin(2 * np.pi * 5 * t2) + 
             np.sin(2 * np.pi * 20 * t2))

# Signal B: 5 Hz for first 2 seconds, THEN 20 Hz for next 2 seconds
# This is like a gel that CHANGES its dominant oscillation as it solidifies!
signal_B = np.concatenate([
    np.sin(2 * np.pi * 5  * t2[:2*fs]),   # first half: only 5 Hz
    np.sin(2 * np.pi * 20 * t2[2*fs:])    # second half: only 20 Hz
])

# Compute FFTs of both
fft_A = (2/len(t2)) * np.abs(fft(signal_A))[:len(t2)//2]
fft_B = (2/len(t2)) * np.abs(fft(signal_B))[:len(t2)//2]
freqs = fftfreq(len(t2), 1/fs)[:len(t2)//2]

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle("⚠️ The FFT Cannot Distinguish These Two Very Different Signals!", 
             fontsize=13, fontweight='bold', color='darkred')

# Time domain plots
axes[0,0].plot(t2, signal_A, color='steelblue', linewidth=0.5)
axes[0,0].set_title('Signal A: 5Hz + 20Hz mixed throughout')
axes[0,0].set_xlabel('Time (s)'); axes[0,0].set_ylabel('Amplitude')

axes[1,0].plot(t2, signal_B, color='darkorange', linewidth=0.5)
axes[1,0].axvline(x=2, color='red', linestyle='--', linewidth=2, label='Transition at t=2s')
axes[1,0].set_title('Signal B: 5Hz for 2s → then 20Hz for 2s (like a gel solidifying!)')
axes[1,0].set_xlabel('Time (s)'); axes[1,0].set_ylabel('Amplitude')
axes[1,0].legend()

# FFT plots — notice they look very similar!
axes[0,1].plot(freqs, fft_A, color='steelblue')
axes[0,1].set_xlim(0, 40); axes[0,1].set_title('FFT of Signal A')
axes[0,1].set_xlabel('Frequency (Hz)')

axes[1,1].plot(freqs, fft_B, color='darkorange')
axes[1,1].set_xlim(0, 40); axes[1,1].set_title('FFT of Signal B — looks almost identical!')
axes[1,1].set_xlabel('Frequency (Hz)')

plt.tight_layout()
plt.show()

print("🚨 KEY INSIGHT: Both signals contain 5 Hz and 20 Hz energy — so their FFTs look similar.")
print("   But Signal B has a critical structural TRANSITION at t=2s that the FFT completely misses.")
print("   In your capstone, this transition = gel point (sol-gel transition during coagulation)!")

---
## 🪟 Part 3: Short-Time Fourier Transform (STFT) — The Band-Aid

### The Fix

Someone in the 1940s had a clever idea: instead of computing FFT over the whole signal, **slide a window** across time and compute FFT inside each window. 

This gives you **frequency content at each time window** — a 2D picture called a **spectrogram**.

### But there's a fundamental trade-off: the Heisenberg Uncertainty Principle for signals

> **You cannot have perfect time resolution AND perfect frequency resolution simultaneously.**

- **Narrow window** → good time resolution, poor frequency resolution (you can tell *when* but not *what frequency*)
- **Wide window** → good frequency resolution, poor time resolution (you know *what frequency* but not *when*)

The window size is **fixed** in STFT — this is its fundamental limitation.

In [ ]:
# ============================================================
# STFT: Sliding window FFT
#
# We'll apply STFT to Signal B (the one with the transition)
# and see how different window sizes affect what we can see.
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('STFT Spectrogram of Signal B — Effect of Window Size', fontsize=13, fontweight='bold')

window_sizes = [64, 512, 2048]
titles = ['Narrow window (64 pts)\nGood time, poor freq', 
          'Medium window (512 pts)\nBalanced', 
          'Wide window (2048 pts)\nPoor time, good freq']
colors = ['RdYlGn', 'RdYlGn', 'RdYlGn']

for ax, nperseg, title in zip(axes, window_sizes, titles):
    f_stft, t_stft, Zxx = signal.stft(signal_B, fs=fs, nperseg=nperseg)
    im = ax.pcolormesh(t_stft, f_stft, np.abs(Zxx), shading='gouraud', cmap='hot_r')
    ax.set_ylim(0, 40)
    ax.axvline(x=2, color='cyan', linestyle='--', linewidth=2, label='True transition')
    ax.set_title(title)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.legend(fontsize=9)
    plt.colorbar(im, ax=ax, label='|Amplitude|')

plt.tight_layout()
plt.show()

print("👆 Observation:")
print("   - Narrow window: can see WHEN the transition happens, but frequencies are blurry")
print("   - Wide window: frequencies are sharp, but the transition looks smeared")
print("   - There's NO window size that gives you both perfectly — this is STFT's fundamental limit.")
print()
print("   For gelation: you need to know BOTH which modulus frequencies are active")
print("   AND exactly WHEN the gel point occurs. STFT forces you to pick one.")

---
## 🌊 Part 4: Wavelet Transform — The Upgrade

### The Big Idea

Instead of using a fixed-size window like STFT, wavelets use a **flexible, self-adapting window**:

- For **high frequencies** → use a **narrow** window (fast oscillations need fine time resolution)
- For **low frequencies** → use a **wide** window (slow oscillations need fine frequency resolution)

This is exactly what our ears do! You can pinpoint a sharp click in time, but you need a longer listening period to distinguish two similar low musical notes.

### How it works

A wavelet is a small, oscillating wave that:
1. Has zero average value (it wiggles up and down)
2. Is localised in time (it decays to zero — unlike a sine wave that goes forever)

The Wavelet Transform slides this little wave across your signal at **multiple scales** (stretched = low frequency, compressed = high frequency) and measures how well it matches.

$$W(a, b) = \frac{1}{\sqrt{a}} \int x(t) \cdot \psi\left(\frac{t-b}{a}\right) dt$$

Where:
- $\psi$ = the mother wavelet (the template shape)
- $a$ = scale (bigger = stretched = lower frequency)
- $b$ = translation (where along time we're looking)

The result is a 2D map called a **scalogram** — showing energy at each scale AND time simultaneously.

In [ ]:
# ============================================================
# CONTINUOUS WAVELET TRANSFORM (CWT)
#
# CWT slides a mother wavelet across the signal at every
# scale and position — giving the richest time-frequency map.
#
# We'll use the Morlet wavelet (best for oscillatory signals
# like rheological measurements).
# ============================================================

# Use a shorter segment for CWT (it's computationally heavy)
# Let's use Signal B — the one with the transition at t=2s
sig = signal_B
dt = 1 / fs  # time step

# Define scales — these correspond to frequencies
# pywt converts scales to pseudo-frequencies for us
scales = np.arange(1, 256)

# Compute CWT using Morlet wavelet ('morl')
coefficients, frequencies = pywt.cwt(sig, scales, 'morl', sampling_period=dt)

# Plot the scalogram
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Continuous Wavelet Transform (CWT) — Morlet Wavelet', fontsize=13, fontweight='bold')

# Time domain signal
axes[0].plot(t2, signal_B, color='purple', linewidth=0.5)
axes[0].axvline(x=2, color='red', linestyle='--', linewidth=2, label='Structural transition')
axes[0].set_title('Original Signal B (5Hz → 20Hz at t=2s)')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')
axes[0].legend()

# CWT scalogram
power = np.abs(coefficients) ** 2
im = axes[1].pcolormesh(t2, frequencies, power, shading='gouraud', cmap='hot_r')
axes[1].axvline(x=2, color='cyan', linestyle='--', linewidth=2, label='True transition (t=2s)')
axes[1].set_ylim(0, 50)
axes[1].set_title('CWT Scalogram — Power at Each Frequency Over Time')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Frequency (Hz)')
axes[1].legend()
plt.colorbar(im, ax=axes[1], label='Power')

plt.tight_layout()
plt.show()

print("✅ CWT correctly shows:")
print("   - 5 Hz energy is STRONG in the first 2 seconds, then disappears")
print("   - 20 Hz energy APPEARS at exactly t=2s")
print("   - The transition is sharp and well-localised in both time AND frequency")
print()
print("   Compare this to the FFT which gave you NO timing information!")

---
## 🏗️ Part 5: Types of Wavelets

Just like there are different tools for different jobs, there are different wavelets for different signals.  
The choice of wavelet matters — here's a practical guide:

| Wavelet Family | Shape | Best For | Relevant to Capstone? |
|---|---|---|---|
| **Morlet** | Oscillating Gaussian | Oscillatory signals, frequency analysis | ✅ Yes — for oscillatory strain signals |
| **Haar** | Step function | Abrupt changes, discontinuities | ✅ Yes — detecting gel point transitions |
| **Daubechies (db)** | Smooth, asymmetric | General purpose, smooth signals | ✅ Yes — general rheological features |
| **Symlets (sym)** | Near-symmetric db | Similar to db but more symmetric | ✅ Possibly |
| **Mexican Hat** | Second derivative of Gaussian | Peak detection, edge detection | ⚠️ Less common for this application |
| **Coiflets** | Symmetric, vanishing moments | Signals with polynomial trends | ⚠️ Possible for baseline drift |


In [ ]:
# ============================================================
# VISUALISING THE WAVELET SHAPES
#
# Each wavelet family has a characteristic shape.
# The shape determines what kinds of features in the signal
# will produce a strong response.
# ============================================================

wavelet_list = [
    ('morl',   'Morlet',         'steelblue',  'Best for oscillatory signals (your rheometer output)'),
    ('haar',   'Haar',           'crimson',    'Best for sudden step changes (gel point detection)'),
    ('db4',    'Daubechies (db4)','darkorange', 'General purpose — smooth, compact support'),
    ('sym4',   'Symlet (sym4)',  'green',      'Near-symmetric Daubechies — less distortion'),
    ('mexh',   'Mexican Hat',   'purple',     'Good for peak/ridge detection'),
    ('coif2',  'Coiflet (coif2)','brown',      'Useful when signal has polynomial trends'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Wavelet Shapes — The "Templates" Used in Analysis', fontsize=14, fontweight='bold')
axes = axes.flatten()

for ax, (name, label, color, note) in zip(axes, wavelet_list):
    try:
        wavelet = pywt.ContinuousWavelet(name) if name == 'morl' or name == 'mexh' else pywt.Wavelet(name)
        if name in ['morl', 'mexh']:
            psi, x = wavelet.wavefun()
            # For continuous wavelets, get real part
            ax.plot(x, np.real(psi), color=color, linewidth=2)
        else:
            phi, psi, x = wavelet.wavefun(level=8)
            ax.plot(x, psi, color=color, linewidth=2)
    except:
        # Fallback for Morlet
        x = np.linspace(-4, 4, 500)
        psi = np.cos(5*x) * np.exp(-0.5*x**2)
        ax.plot(x, psi, color=color, linewidth=2)
    
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_xlabel('Time'); ax.set_ylabel('Amplitude')
    ax.axhline(0, color='gray', linewidth=0.5)
    
    # Add note as text box
    ax.text(0.5, -0.25, note, transform=ax.transAxes, ha='center', 
            fontsize=8, style='italic', wrap=True)

plt.tight_layout(pad=3.0)
plt.show()

print("💡 Rule of thumb: choose a wavelet that 'looks like' what you're trying to detect in your signal.")
print("   Morlet is most common for oscillatory rheological signals (like oscillatory shear experiments).")
print("   Haar is great for detecting the sol-gel transition (a sudden change in elastic behaviour).")

---
## 🔀 Part 6: Discrete Wavelet Transform (DWT) — The Efficient Version

CWT is beautiful but **very slow** — it computes coefficients at every possible scale and position.

**DWT** is a faster, more practical version that:
- Uses **dyadic scales** (scale = 1, 2, 4, 8, 16... — doubling each time)
- Decomposes your signal into a **hierarchy of levels** (approximation + details)
- Is implemented as a fast filterbank — extremely efficient

### Multi-Level Decomposition (like peeling an onion)

```
Signal
  ├── Approximation (low freq, A1)  ──→  Approximation (A2) ──→ A3...
  └── Detail (high freq, D1)            └── Detail (D2)
```

Each level splits the signal into:
- **Approximations (A)**: the slow, smooth part (like the overall modulus trend)
- **Details (D)**: the fast, fine part (like harmonic noise or sharp transitions)

In [ ]:
# ============================================================
# DISCRETE WAVELET TRANSFORM (DWT) — MULTI-LEVEL DECOMPOSITION
#
# Let's decompose a simulated 'gelation' signal into
# approximation + detail levels.
# ============================================================

# Simulate a more realistic gelation signal:
# - Slowly increasing baseline (gel stiffening) = low frequency trend
# - Oscillatory rheometer drive signal = medium frequency
# - Measurement noise = high frequency
# - Sharp transition at t=30s (the gel point)

np.random.seed(42)
t_gel = np.linspace(0, 60, 6000)   # 60 second experiment, 100 Hz

# Components of the simulated gelation signal
baseline_trend = 0.5 * (1 - np.exp(-t_gel / 25))   # slow exponential rise (gel stiffening)
oscillation    = 0.1 * np.sin(2 * np.pi * 1 * t_gel)  # 1 Hz rheometer oscillation
noise          = 0.02 * np.random.randn(len(t_gel))    # instrument noise
gel_point_jump = 0.2 * (t_gel > 30).astype(float)     # structural transition at 30s

gelation_signal = baseline_trend + oscillation + noise + gel_point_jump

# Apply DWT — decompose to 4 levels using Daubechies 4
wavelet = 'db4'
level = 4
coeffs = pywt.wavedec(gelation_signal, wavelet, level=level)

# Reconstruct each level separately for visualisation
# This shows what each level 'sees' in the signal
fig, axes = plt.subplots(level + 2, 1, figsize=(14, 14), sharex=False)
fig.suptitle('DWT Multi-Level Decomposition of Simulated Gelation Signal (db4)', 
             fontsize=13, fontweight='bold')

# Original signal
axes[0].plot(t_gel, gelation_signal, color='purple', linewidth=0.8)
axes[0].axvline(30, color='red', linestyle='--', linewidth=1.5, label='Gel point (t=30s)')
axes[0].set_title('Original Gelation Signal')
axes[0].set_ylabel('G\' (arb.)'); axes[0].legend()

labels = [f'A{level} — Approximation (slowest trends, ~DC+very low freq)',
          f'D{level} — Detail Level 4 (low-mid freq)',
          f'D{level-1} — Detail Level 3 (mid freq)',
          f'D{level-2} — Detail Level 2 (mid-high freq)',
          f'D1 — Detail Level 1 (highest freq — noise)']
plot_colors = ['steelblue', 'darkorange', 'green', 'crimson', 'gray']

for i, (coeff, lbl, col) in enumerate(zip(coeffs, labels, plot_colors)):
    t_sub = np.linspace(0, 60, len(coeff))
    axes[i+1].plot(t_sub, coeff, color=col, linewidth=0.9)
    axes[i+1].set_title(lbl, fontsize=10)
    axes[i+1].set_ylabel('Coeff.')
    if i == 0:
        axes[i+1].axvline(30, color='red', linestyle='--', linewidth=1)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

print("💡 What to notice:")
print("   - A4 (approximation): captures the slow exponential rise — the gel stiffening trend")
print("   - D4, D3: capture the rheometer oscillation")
print("   - D1 (finest detail): mostly just noise")
print("   - The gel point jump is visible as a spike in some detail levels at t=30s")

---
## 🔬 Part 7: Connecting Everything to Our Capstone

### What your rheometer actually measures

In an oscillatory shear experiment, the rheometer applies a sinusoidal strain $\gamma(t) = \gamma_0 \sin(\omega t)$  
and measures the resulting stress $\sigma(t)$.

From this, you extract:
- **G' (storage modulus)** — elastic (solid-like) behaviour
- **G'' (loss modulus)** — viscous (liquid-like) behaviour
- **tan δ = G''/G'** — the phase angle

During gelation, **G' crosses G''** — this crossover is the **gel point**.  
Wavelets can detect this crossover far more precisely than FFT.

### The wavelet features we weill extract (from the project brief)

| Feature | What it captures | Wavelet approach |
|---|---|---|
| **Scale drift** | How dominant frequency shifts as gel forms | CWT scalogram centroid over time |
| **Harmonic energy redistribution** | Energy moving between harmonics | DWT level energies over time |
| **Entropy measures** | Disorder → order during gelation | Shannon entropy of wavelet coefficients |
| **Instantaneous modulus** | G' and G'' at each moment | CWT coefficient magnitudes |

In [ ]:
# ============================================================
# CAPSTONE APPLICATION: Detecting the Gel Point
#
# We'll simulate a realistic G'(t) signal during gelation
# and compare how FFT vs Wavelet methods detect the gel point.
# ============================================================

np.random.seed(0)
fs_rheo = 10   # 10 Hz sampling — typical for rheometer
t_rheo = np.linspace(0, 120, 120 * fs_rheo)  # 2-minute experiment

# Simulate G' during gelation:
# Phase 1 (0-60s): liquid-like, G' low and slowly rising
# Phase 2 (60-120s): gel forming rapidly, G' rises steeply
gel_point_t = 60  # seconds

def simulate_G_prime(t, gel_point=60):
    """Simulate storage modulus G' during gelation"""
    # Sigmoid-like rise centred at gel point
    G_prime = 10 + 200 / (1 + np.exp(-0.15 * (t - gel_point)))
    # Add oscillatory noise (the rheometer drive)
    G_prime += 5 * np.sin(2 * np.pi * 0.5 * t)
    # Add measurement noise
    G_prime += np.random.normal(0, 2, len(t))
    return G_prime

def simulate_G_double_prime(t, gel_point=60):
    """Simulate loss modulus G'' during gelation"""
    G_dp = 50 + 30 * np.exp(-0.03 * (t - gel_point)**2)  # peaks near gel point
    G_dp += np.random.normal(0, 2, len(t))
    return G_dp

G_prime = simulate_G_prime(t_rheo, gel_point_t)
G_double_prime = simulate_G_double_prime(t_rheo, gel_point_t)

# --- DWT-based analysis: energy in each level over time ---
# We slide a window and compute DWT energy at each window
window_size = 50   # 5 second window
step = 5           # step 0.5 seconds
times_win = []
energies_A = []    # approximation energy (slow trend)
energies_D1 = []   # finest detail energy (noise/high freq content)

for start in range(0, len(G_prime) - window_size, step):
    window = G_prime[start:start + window_size]
    c = pywt.wavedec(window, 'db4', level=3)
    energies_A.append(np.sum(c[0]**2))   # approximation energy
    energies_D1.append(np.sum(c[-1]**2)) # detail level 1 energy
    times_win.append(t_rheo[start + window_size // 2])

times_win = np.array(times_win)
energies_A = np.array(energies_A)
energies_D1 = np.array(energies_D1)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 11))
fig.suptitle('Capstone Application: Wavelet Analysis of Simulated Gelation', 
             fontsize=13, fontweight='bold')

# G' and G'' over time
axes[0].plot(t_rheo, G_prime, label="G' (Storage Modulus)", color='steelblue', linewidth=1)
axes[0].plot(t_rheo, G_double_prime, label="G'' (Loss Modulus)", color='darkorange', linewidth=1)
axes[0].axvline(gel_point_t, color='red', linestyle='--', linewidth=2, label='True Gel Point (t=60s)')

# Find G' = G'' crossover numerically
diff = G_prime - G_double_prime
crossover_idx = np.where(np.diff(np.sign(diff)))[0]
if len(crossover_idx) > 0:
    t_crossover = t_rheo[crossover_idx[0]]
    axes[0].axvline(t_crossover, color='purple', linestyle=':', linewidth=2, 
                    label=f"G'=G'' crossover (t={t_crossover:.1f}s)")

axes[0].set_ylabel('Modulus (Pa)')
axes[0].set_title("Simulated G' and G'' During Gelation")
axes[0].legend(fontsize=9)

# DWT approximation energy (captures the slow stiffening trend)
axes[1].plot(times_win, energies_A / energies_A.max(), color='steelblue', linewidth=1.5,
             label='Approximation Energy (normalised) — slow trend')
axes[1].axvline(gel_point_t, color='red', linestyle='--', linewidth=2, label='True Gel Point')
axes[1].set_ylabel('Normalised Energy')
axes[1].set_title('DWT Approximation Energy — Tracks G\' Rise During Gelation')
axes[1].legend(fontsize=9)

# DWT detail energy (captures noise/structural fluctuations)
axes[2].plot(times_win, energies_D1 / energies_D1.max(), color='crimson', linewidth=1.5,
             label='Detail Level 1 Energy (normalised) — fine structure')
axes[2].axvline(gel_point_t, color='red', linestyle='--', linewidth=2, label='True Gel Point')
axes[2].set_ylabel('Normalised Energy')
axes[2].set_xlabel('Time (s)')
axes[2].set_title('DWT Detail Energy — Peaks Near Gel Point (Structural Fluctuations)')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("✅ Wavelet-derived features can track the gelation process in real time:")
print("   - Approximation energy tracks the overall stiffness increase (G' rise)")
print("   - Detail energy captures structural fluctuations around the gel point")
print("   - Together, these give a richer picture than FFT's single frequency snapshot")

In [ ]:
# ============================================================
# WAVELET ENTROPY — Measuring Order vs Disorder
#
# Shannon entropy of wavelet coefficients measures
# how 'spread out' the energy is across scales.
#
# BEFORE gelation: energy spread across many scales (disordered liquid)
# AFTER gelation:  energy concentrated in few scales (ordered solid)
# => Entropy DECREASES at the gel point — a powerful marker!
# ============================================================

def wavelet_shannon_entropy(signal_window, wavelet='db4', level=3):
    """Compute normalised Shannon entropy of wavelet coefficient energies."""
    coeffs = pywt.wavedec(signal_window, wavelet, level=level)
    energies = np.array([np.sum(c**2) for c in coeffs])
    total = energies.sum()
    if total == 0:
        return 0
    probs = energies / total           # normalise to probability distribution
    # Shannon entropy: H = -sum(p * log(p))
    entropy = -np.sum(probs * np.log(probs + 1e-12))
    return entropy

# Compute entropy across sliding windows
entropies = []
times_ent = []
for start in range(0, len(G_prime) - window_size, step):
    window = G_prime[start:start + window_size]
    ent = wavelet_shannon_entropy(window)
    entropies.append(ent)
    times_ent.append(t_rheo[start + window_size // 2])

entropies = np.array(entropies)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(times_ent, entropies, color='darkorchid', linewidth=2, label='Wavelet Shannon Entropy')
ax.axvline(gel_point_t, color='red', linestyle='--', linewidth=2, label='True Gel Point (t=60s)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Shannon Entropy of Wavelet Coefficients')
ax.set_title('Wavelet Entropy as a Gelation Marker — Entropy Drops at Gel Point', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("✅ Shannon entropy of wavelet coefficients is a powerful structural transition marker.")
print("   - In a liquid: energy distributed across many scales → HIGH entropy")
print("   - At/after gel point: energy concentrates → LOWER entropy")
print("   This is one of the 'wavelet-derived features' mentioned in your project brief!")

---
## 📊 Part 8: FFT vs STFT vs CWT — Side by Side Comparison

Let's put everything together and compare the three methods on the same gelation signal.

In [ ]:
# ============================================================
# FINAL COMPARISON: FFT vs STFT vs CWT on a gelation signal
# ============================================================

# Use G_prime signal from above
sig_compare = G_prime
dt_rheo = 1 / fs_rheo

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.5, wspace=0.35)
fig.suptitle("FFT vs STFT vs CWT on Simulated Gelation Signal", fontsize=14, fontweight='bold')

# 1. Original signal
ax_sig = fig.add_subplot(gs[0, :])
ax_sig.plot(t_rheo, sig_compare, color='purple', linewidth=0.8)
ax_sig.axvline(gel_point_t, color='red', linestyle='--', linewidth=2, label='Gel Point')
ax_sig.set_title("Simulated G'(t) during gelation"); ax_sig.set_xlabel('Time (s)'); ax_sig.set_ylabel("G' (Pa)")
ax_sig.legend()

# 2. FFT
ax_fft = fig.add_subplot(gs[1, 0])
N = len(sig_compare)
yf = (2/N) * np.abs(fft(sig_compare))[:N//2]
xf = fftfreq(N, dt_rheo)[:N//2]
ax_fft.plot(xf, yf, color='crimson', linewidth=1.5)
ax_fft.set_xlim(0, 3)
ax_fft.set_title('FFT — Frequency only, no time info\n⚠️ Cannot detect when gel point occurs')
ax_fft.set_xlabel('Frequency (Hz)'); ax_fft.set_ylabel('Amplitude')
ax_fft.text(0.5, 0.85, '❌ No timing information', transform=ax_fft.transAxes, 
            ha='center', color='darkred', fontsize=10, fontweight='bold')

# 3. STFT
ax_stft = fig.add_subplot(gs[1, 1])
f_stft, t_stft, Zxx = signal.stft(sig_compare, fs=fs_rheo, nperseg=50)
ax_stft.pcolormesh(t_stft, f_stft, np.abs(Zxx), shading='gouraud', cmap='hot_r')
ax_stft.axvline(gel_point_t, color='cyan', linestyle='--', linewidth=2)
ax_stft.set_ylim(0, 3)
ax_stft.set_title('STFT — Some time-frequency info\n⚠️ Fixed window limits resolution')
ax_stft.set_xlabel('Time (s)'); ax_stft.set_ylabel('Frequency (Hz)')
ax_stft.text(0.5, 0.85, '⚠️ Fixed resolution trade-off', transform=ax_stft.transAxes,
             ha='center', color='darkorange', fontsize=10, fontweight='bold')

# 4. CWT
ax_cwt = fig.add_subplot(gs[2:, :])
scales_rheo = np.arange(1, 64)
coeff_cwt, freq_cwt = pywt.cwt(sig_compare, scales_rheo, 'morl', sampling_period=dt_rheo)
power_cwt = np.abs(coeff_cwt) ** 2
im = ax_cwt.pcolormesh(t_rheo, freq_cwt, power_cwt, shading='gouraud', cmap='hot_r')
ax_cwt.axvline(gel_point_t, color='cyan', linestyle='--', linewidth=2.5, label='Gel Point')
ax_cwt.set_ylim(0, 3)
ax_cwt.set_title('CWT (Morlet) — Adaptive time-frequency resolution\n✅ Can see gel point AND frequency evolution together',
                 fontweight='bold')
ax_cwt.set_xlabel('Time (s)'); ax_cwt.set_ylabel('Frequency (Hz)')
ax_cwt.legend(fontsize=10)
ax_cwt.text(0.5, 0.9, '✅ Best of both worlds', transform=ax_cwt.transAxes,
            ha='center', color='lime', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax_cwt, label='Power')

plt.show()

print("="*65)
print("SUMMARY COMPARISON")
print("="*65)
print(f"{'Method':<10} {'Time Info':<15} {'Freq Info':<15} {'For Evolving Systems'}")
print("-"*65)
print(f"{'FFT':<10} {'❌ None':<15} {'✅ High':<15} {'❌ Poor'}")
print(f"{'STFT':<10} {'⚠️ Fixed':<15} {'⚠️ Fixed':<15} {'⚠️ OK but limited'}")
print(f"{'CWT':<10} {'✅ Adaptive':<15} {'✅ Adaptive':<15} {'✅ Excellent'}")
print(f"{'DWT':<10} {'✅ Fast':<15} {'✅ Multi-level':<15} {'✅ Excellent + fast'}")

---
## 🗺️ Part 9: Summary — Our Capstone Roadmap

Here's how everything maps to the Haemograph project deliverables:

### Project deliverable → Wavelet tool

| Deliverable (from brief) | What to do | Wavelet tool |
|---|---|---|
| Compare FFT vs Wavelet | Show FFT's time-blindness vs CWT's scalogram | FFT + CWT side by side |
| Extract instantaneous modulus evolution | Track G', G'' at each timepoint | CWT magnitude at drive frequency |
| Harmonic content analysis | Track energy at harmonics of drive frequency | DWT level energies |
| Structural transition markers | Detect gel point automatically | DWT entropy, CWT ridge tracking |
| Scale drift | How dominant scale shifts during gelation | CWT scalogram centroid over time |
| Relate to gelation kinetics | Correlate wavelet features with known gel chemistry | Feature extraction + correlation analysis |

### Recommended wavelet choices for this project

- **CWT with Morlet**: for oscillatory rheological data (best for frequency-time mapping)
- **DWT with db4 or db6**: for energy decomposition and entropy calculation (fast, practical)
- **DWT with Haar**: for detecting sharp transitions (gel point)

---

## 🔑 Key Takeaways

1. **FFT** tells you *what* frequencies exist globally — useless for evolving systems
2. **STFT** adds some time info but is limited by the fixed window size (resolution trade-off)
3. **CWT** gives you the full time-frequency picture with adaptive resolution — the gold standard for analysis
4. **DWT** is the fast, practical version — great for feature extraction and real-time processing
5. **Wavelet entropy** is a powerful gel point marker — entropy drops as the gel becomes ordered
6. **Choice of wavelet shape** matters — Morlet for oscillatory signals, Haar for sharp transitions

---
